In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import json
import os
import urllib
import ssl

def download_and_load_file(file_path, url):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context=ssl_context) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

In [ ]:
def format_input(entry):
    instruction_text=(
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Input:\n{entry['instruction']}"
    )
    
    input_text=f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text+input_text

In [ ]:
train_ratio=0.85
test_ratio=0.10
train_portion=int(len(data)*train_ratio)
test_portion=int(len(data)*test_ratio)
train_data=data[:train_portion]
test_data=data[train_portion:train_portion+test_portion]
val_data=data[train_portion+test_portion:]

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self,data,tokenizer):
        self.data=data
        self.encoded_data=[]
        for entry in data:
            instruction_plus_input=format_input(entry)
            response_text=f"\n\n### Response:\n{entry['output']}"
            full_text=instruction_plus_input+response_text
            self.encoded_data.append(tokenizer.encode(full_text))
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        return self.encoded_data[index]
    
        

In [ ]:
def custom_collate_function(batch,pad_token_id=50256,ignore_index=-100,allowed_max_length=None,device='cpu'):
    batch_max_length=max(len(item)+1 for item in batch)
    input_lst,target_lst=[],[]
    for item in batch:
        new_item=item.copy()
        new_item+=[pad_token_id]
        padded=(new_item+ [pad_token_id]*(batch_max_length-len(new_item)))
        inputs=torch.tensor(padded[:-1])
        outputs=torch.tensor(padded[1:])
        mask=(outputs==pad_token_id)
        indices=torch.nonzero(mask).squeeze()
        if indices.numel()>1:
            outputs[indices[1:]]=ignore_index
        if allowed_max_length is not None:
            inputs=inputs[:allowed_max_length]
            outputs=outputs[:allowed_max_length]
        input_lst.append(inputs)
        target_lst.append(outputs)
    inputs_tensor = torch.stack(input_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor,targets_tensor

In [ ]:
def create_dataloader(data,batch_size,collate_fn,tokenizer,drop_last):
    dataset=InstructionDataset(data,tokenizer)
    data_loaded=DataLoader(dataset,batch_size=batch_size,collate_fn=collate_fn,shuffle=False,drop_last=drop_last)
    return data_loaded

In [ ]:
!pip3 install tiktoken
import tiktoken

In [ ]:
tokenizer=tiktoken.get_encoding('gpt2')

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from functools import partial
collate_fn=partial(custom_collate_function,device=device,allowed_max_length=1024)

In [ ]:
batch_size=8
train_loader=create_dataloader(train_data,batch_size=batch_size,collate_fn=collate_fn,tokenizer=tokenizer,drop_last=True)
val_loader=create_dataloader(val_data,batch_size=batch_size,collate_fn=collate_fn,tokenizer=tokenizer,drop_last=False)
test_loader=create_dataloader(test_data,batch_size=batch_size,collate_fn=collate_fn,tokenizer=tokenizer,drop_last=False)

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/adnanik23/input-new')

In [ ]:
from gpt2_architecture import GPTModel

In [ ]:
from  gpt_download3 import download_and_load_gpt2

In [ ]:
base_config={"vocab_size":50257,"context_length":1024,"embed_dim":768,"num_heads":12,"n_layers":12,"dropout":0.0,"qkv_bias":True}

In [ ]:
base_config.update({'embed_dim':1024,'num_heads':16,'n_layers':24})

In [ ]:
settings,params=download_and_load_gpt2(model_size="355M",models_dir="gpt2")

In [ ]:
from gpt2_architecture import load_weights_into_model

In [ ]:
model=GPTModel(base_config)
load_weights_into_model(model,params)

In [ ]:
model.eval()

In [ ]:
model.to(device)

In [ ]:
from gpt2_architecture import generate_new_text,text_to_token_ids,token_ids_to_text,train_model

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0005,weight_decay=0.1)
num_epochs=5
train_losses,val_losses=train_model(model,train_loader,val_loader,optimizer,device=device,num_epochs=num_epochs,eval_freq=5,eval_iter=5)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, train_losses, val_losses)